In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

path = '/content/drive/MyDrive/churn-prediction-project/processed'

X_train = pd.read_csv(f'{path}/X_train.csv')
X_test = pd.read_csv(f'{path}/X_test.csv')
y_train = pd.read_csv(f'{path}/y_train.csv').squeeze()
y_test = pd.read_csv(f'{path}/y_test.csv').squeeze()

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print(y_train.value_counts())

Mounted at /content/drive
X_train: (5634, 31) X_test: (1409, 31)
Churn Value
0    4139
1    1495
Name: count, dtype: int64


In [2]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train, y_train)

print("Model trained!")

Model trained!


In [3]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = log_reg.predict(X_test)
y_pred_proba = log_reg.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nROC-AUC Score:", round(roc_auc_score(y_test, y_pred_proba), 4))

Confusion Matrix:
[[759 276]
 [ 84 290]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1035
           1       0.51      0.78      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.75      0.71      1409
weighted avg       0.80      0.74      0.76      1409


ROC-AUC Score: 0.8482


In [5]:
#Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba_rf), 4))

Random Forest
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1035
           1       0.64      0.49      0.55       374

    accuracy                           0.79      1409
   macro avg       0.73      0.69      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC: 0.8399


In [6]:
#XGBoost
import xgboost as xgb

# scale_pos_weight is XGBoost's version of class_weight='balanced'
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost")
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba_xgb), 4))

XGBoost
              precision    recall  f1-score   support

           0       0.86      0.83      0.85      1035
           1       0.58      0.64      0.61       374

    accuracy                           0.78      1409
   macro avg       0.72      0.74      0.73      1409
weighted avg       0.79      0.78      0.79      1409

ROC-AUC: 0.8265


Surprise: Logistic Regression actually wins on the two metrics that matter most for a churn use case — it has the highest ROC-AUC and by far the best recall (catches 78% of customers who actually churn, vs. 49-64% for the tree models). Random Forest has the best accuracy and precision, but it's missing over half the customers who actually leave — for a retention business problem, that's a worse outcome even though "accuracy" looks better.
This is actually a great insight for your README: "Logistic Regression outperformed Random Forest and XGBoost on ROC-AUC and recall for the churn class, likely because class_weight='balanced' directly optimizes for the minority class, while the tree models would need hyperparameter tuning (e.g., via GridSearchCV) to match this without it." That sentence alone shows you understand model selection isn't just "pick the fanciest algorithm" — which is exactly the kind of thing that stands out for a fresher.
Let's go with Logistic Regression as your final model. Now let's get feature importance (great for your dashboard and README) and save everything for deployment.

In [7]:
coefficients = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': log_reg.coef_[0]
}).sort_values('coefficient', ascending=False)

print("Top 10 features INCREASING churn risk:")
print(coefficients.head(10))

print("\nTop 10 features DECREASING churn risk:")
print(coefficients.tail(10))

Top 10 features INCREASING churn risk:
                            feature  coefficient
2                     Total Charges     1.089456
11     Internet Service_Fiber optic     0.738777
9   Multiple Lines_No phone service     0.420641
29  Payment Method_Electronic check     0.397417
27            Paperless Billing_Yes     0.315287
6                       Partner_Yes     0.265232
10               Multiple Lines_Yes     0.257485
24             Streaming Movies_Yes     0.249740
22                 Streaming TV_Yes     0.229837
3                              CLTV     0.087047

Top 10 features DECREASING churn risk:
                                  feature  coefficient
19       Tech Support_No internet service    -0.108227
17  Device Protection_No internet service    -0.108227
21       Streaming TV_No internet service    -0.108227
16                      Online Backup_Yes    -0.127697
20                       Tech Support_Yes    -0.336455
14                    Online Security_Yes    -0.3780

In [8]:
import joblib

joblib.dump(log_reg, f'{path}/churn_model.pkl')
joblib.dump(X_train.columns.tolist(), f'{path}/model_columns.pkl')

print("Model and column list saved!")

Model and column list saved!


In [13]:
import sqlite3

conn = sqlite3.connect('/content/drive/MyDrive/churn-prediction-project/churn.db')

In [15]:
scaler = joblib.load(f'{path}/scaler.pkl')

In [16]:
import os

# Reload the original (readable) data, fixed
df_full = pd.read_sql("SELECT * FROM customers", conn) if conn else None
df_full['Total Charges'] = pd.to_numeric(df_full['Total Charges'], errors='coerce').fillna(0)

# Recreate features exactly as during training
cols_to_drop = ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
                 'Lat Long', 'Latitude', 'Longitude',
                 'Churn Label', 'Churn Score', 'Churn Reason', 'Churn Value']
cols_to_drop = [c for c in cols_to_drop if c in df_full.columns]
features_full = df_full.drop(columns=cols_to_drop)

categorical_cols = features_full.select_dtypes(include='object').columns.tolist()
numeric_cols = ['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']

X_full = pd.get_dummies(features_full, columns=categorical_cols, drop_first=True)
X_full = X_full.reindex(columns=X_train.columns, fill_value=0)
X_full[numeric_cols] = scaler.transform(X_full[numeric_cols])

# Predict churn probability for EVERY customer
df_full['Churn_Probability'] = log_reg.predict_proba(X_full)[:, 1]
df_full['Risk_Level'] = pd.cut(df_full['Churn_Probability'], bins=[0, 0.3, 0.6, 1.0], labels=['Low', 'Medium', 'High'])

# Save for Power BI
dashboard_path = '/content/drive/MyDrive/churn-prediction-project/dashboard'
os.makedirs(dashboard_path, exist_ok=True)
df_full.to_csv(f'{dashboard_path}/dashboard_data.csv', index=False)

# Also save feature importance as its own small file
coefficients.to_csv(f'{dashboard_path}/feature_importance.csv', index=False)

print("Exported:", df_full.shape)
print(df_full[['Churn Label', 'Churn_Probability', 'Risk_Level']].head())

Exported: (7043, 35)
  Churn Label  Churn_Probability Risk_Level
0         Yes           0.565909     Medium
1         Yes           0.551300     Medium
2         Yes           0.687628       High
3         Yes           0.542924     Medium
4         Yes           0.297837        Low
